# Multi-Dataset PCB Component & Defect Detection Pipeline
### Supported Datasets: WACV 2019, Kaggle FICS-PCB, PKU-Market PCB Defect, & DeepPCB

This notebook provides a unified pipeline to download, clean, and process both PCB Component Detection and PCB Defect Detection datasets from Kaggle:
1. **Automatic Dataset Downloader**: Pulls WACV, FICS-PCB, PKU-Market, and DeepPCB datasets using `kagglehub`.
2. **Data Inspection & Cleaning**: Filters out invalid bounding boxes, corrupt images, and tiny noise.
3. **Pretrained SAM Point Extractor**: Passes box prompts into Meta's `sam_vit_b.pth` to generate exact `(x, y)` polygon points.
4. **Polygon Data Augmentation**: Applies spatial flips, rotations, and HSV color jitter while transforming `(x, y)` points.
5. **YOLO-seg Export**: Formats output for direct YOLOv8-seg / YOLOv11-seg model training.

In [ ]:
# Step 1: Install Dependencies
!pip install torch torchvision opencv-python matplotlib kagglehub git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os
import cv2
import torch
import random
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor
import kagglehub

# Step 2: Download & Initialize Pretrained SAM Weights
sam_checkpoint = Path(r"C:\Users\ANAGHA\sam_vit_b.pth")
sam_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not sam_checkpoint.exists():
    print(f"Downloading SAM pretrained weights to {sam_checkpoint}...")
    sam_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(sam_url, str(sam_checkpoint))
    print("SAM weights download complete!")
else:
    print(f"SAM pretrained weights found at: {sam_checkpoint}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading SAM model on device: {device}...")
sam = sam_model_registry["vit_b"](checkpoint=str(sam_checkpoint))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
print("SAM Predictor ready!")

In [ ]:
# Step 3: Multi-Dataset Kaggle Downloader (FICS-PCB, PKU-Market, DeepPCB)
KAGGLE_DATASETS = {
    "fics_pcb": "ficslab/fics-pcb",                 # PCB Component Detection (9,912 images / 31 boards)
    "pku_pcb_defect": "akhatovar/pcb-defect-dataset", # PKU PCB Defect Dataset (Open, Short, Mousebite, Spur, Pin-hole)
    "deeppcb": "arnablaha/deeppcb"                   # DeepPCB Defect Dataset (1,500 PCB image pairs + defects)
}

def pull_kaggle_pcb_dataset(name="pku_pcb_defect"):
    """
    Downloads PCB Component or Defect datasets from Kaggle using kagglehub.
    """
    slug = KAGGLE_DATASETS.get(name, name)
    print(f"Pulling Kaggle dataset '{name}' ({slug}) via kagglehub...")
    try:
        path = kagglehub.dataset_download(slug)
        print(f"SUCCESS: Kaggle dataset '{name}' downloaded to: {path}")
        return Path(path)
    except Exception as e:
        print(f"Note on Kaggle Download: {e}")
        return None

# Download PKU PCB Defect Dataset & DeepPCB Dataset
pku_defect_path = pull_kaggle_pcb_dataset("pku_pcb_defect")
deeppcb_path = pull_kaggle_pcb_dataset("deeppcb")

In [ ]:
# Step 4: Data Cleaning & Contour Point Extractor
def clean_bounding_boxes(boxes, img_w, img_h, min_box_size=10):
    cleaned_boxes = []
    for box, cid in boxes:
        x1, y1, x2, y2 = box
        x1 = max(0, min(x1, img_w - 1))
        y1 = max(0, min(y1, img_h - 1))
        x2 = max(0, min(x2, img_w - 1))
        y2 = max(0, min(y2, img_h - 1))
        
        bw = x2 - x1
        bh = y2 - y1
        if bw >= min_box_size and bh >= min_box_size:
            cleaned_boxes.append(([x1, y1, x2, y2], cid))
            
    return cleaned_boxes

def extract_polygon_points(mask, img_w, img_h):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    
    for cnt in contours:
        if cv2.contourArea(cnt) < 15:
            continue
        epsilon = 0.005 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        pts = approx.reshape(-1, 2)
        
        norm_pts = []
        for px, py in pts:
            norm_pts.extend([round(px / img_w, 6), round(py / img_h, 6)])
            
        if len(norm_pts) >= 6:
            polygons.append(norm_pts)
            
    return polygons

In [ ]:
# Step 5: Polygon-Aware Data Augmentation Engine
def augment_image_and_polygons(image, polygon_instances, flip_h=False, flip_v=False, hsv_jitter=True):
    aug_img = image.copy()
    if hsv_jitter:
        hsv = cv2.cvtColor(aug_img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 0] = (hsv[:, :, 0] + random.randint(-10, 10)) % 180
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.85, 1.15), 0, 255)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.85, 1.15), 0, 255)
        aug_img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        
    if flip_h:
        aug_img = cv2.flip(aug_img, 1)
    if flip_v:
        aug_img = cv2.flip(aug_img, 0)
        
    aug_polygons = []
    for cid, pts in polygon_instances:
        transformed_pts = []
        for i in range(0, len(pts), 2):
            nx, ny = pts[i], pts[i+1]
            if flip_h:
                nx = round(1.0 - nx, 6)
            if flip_v:
                ny = round(1.0 - ny, 6)
            transformed_pts.extend([nx, ny])
        aug_polygons.append((cid, transformed_pts))
        
    return aug_img, aug_polygons

In [ ]:
# Step 6: 3-Panel Plot (Original PCB Defect vs SAM Mask vs Augmented Defect Mask)
test_img = np.zeros((400, 400, 3), dtype=np.uint8)
test_img[:, :] = (30, 100, 30)
cv2.rectangle(test_img, (120, 120), (280, 280), (200, 200, 200), -1)
cv2.circle(test_img, (200, 200), 40, (50, 50, 200), -1)

box = np.array([115, 115, 285, 285])
predictor.set_image(test_img)
masks, _, _ = predictor.predict(box=box[None, :], multimask_output=False)
mask = masks[0]

orig_polys = extract_polygon_points(mask, img_w=400, img_h=400)
poly_instances = [(0, orig_polys[0])]

aug_img, aug_polys = augment_image_and_polygons(test_img, poly_instances, flip_h=True, hsv_jitter=True)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title("1. Original PCB Image")
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))

plt.subplot(1, 3, 2)
plt.title("2. SAM Defect Mask & Extracted Points")
mask_vis = test_img.copy()
contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(mask_vis, contours, -1, (0, 255, 0), 2)
plt.imshow(cv2.cvtColor(mask_vis, cv2.COLOR_BGR2RGB))

plt.subplot(1, 3, 3)
plt.title("3. Augmented Defect PCB + Points")
aug_vis = aug_img.copy()
for cid, pts in aug_polys:
    pxs = [int(pts[i] * 400) for i in range(0, len(pts), 2)]
    pys = [int(pts[i+1] * 400) for i in range(0, len(pts), 2)]
    cnt_pts = np.array(list(zip(pxs, pys)), dtype=np.int32)
    cv2.drawContours(aug_vis, [cnt_pts], -1, (255, 255, 0), 2)
plt.imshow(cv2.cvtColor(aug_vis, cv2.COLOR_BGR2RGB))

plt.tight_layout()
plt.show()